# QCNN 手写数字二分类 demo

本 Notebook 在 `sklearn.datasets.load_digits` 数据集上做 0/1 二分类，演示一个端到端的量子机器学习流程：

1. 数据加载与可视化
2. 数据预处理：8x8 图像 → 2x4 平均池化 → 8 维 → 缩放到 [-π, π]
3. 8 比特 QCNN 电路构建（卷积层 + 池化层）
4. 训练（SPSA + 实时损失曲线）
5. 评估（混淆矩阵 + 错分样本）
6. 模型保存与加载
7. 经典 SVM / MLP 基线对比
8. 总结

**预计训练时间**：CPU 上约 5–8 分钟（150 次 SPSA 迭代）。

**依赖**：先在仓库根目录执行 `pip install -e .`，再 `pip install matplotlib seaborn jupyter pylatexenc`。


## 0. 环境准备

固定随机种子，把当前目录加入 `sys.path` 以便 `import src.xxx`。

In [ ]:
import sys, os, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# 让 notebook 能 import 到 src/
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

from qiskit_machine_learning.utils import algorithm_globals
SEED = 42
algorithm_globals.random_seed = SEED
np.random.seed(SEED)

print("环境就绪，输出目录:", OUT_DIR)


## 1. 数据加载与可视化

`sklearn.datasets.load_digits()` 提供了 1797 张 8×8 的灰度手写数字图。
我们只取 label 为 0 或 1 的子集（约 360 张），做 80/20 的 train/test split。


In [ ]:
from src.data import load_and_prepare_data
from src import viz

# mode='spatial': 8x8 图像通过 4x2 平均池化 -> 2x4 -> 展平 8 维。
#                 这种"空间下采样"方式保留了图像的局部结构，是 QCNN 卷积层的前提。
# mode='pca':     PCA 64 -> 8。抽象但破坏了空间局部性，QCNN 难以收敛。
data = load_and_prepare_data(classes=(0, 1), test_size=0.2, seed=SEED, mode="spatial")

print(f"模式: {data.mode}")
print(f"训练集 {data.x_train.shape[0]} 样本，测试集 {data.x_test.shape[0]} 样本")
print(f"特征维度: {data.x_train.shape[1]}")
print(f"特征范围: [{data.x_train.min():.3f}, {data.x_train.max():.3f}]")

viz.plot_samples(data.raw_train_images, data.y_train, n=8, out_path=OUT_DIR / "samples.png")
from IPython.display import Image
Image(filename=str(OUT_DIR / "samples.png"))


## 2. 数据预处理可视化

下面把"原始 8×8 图"和"2×4 池化后的图"放在一起对比。可以看到池化后保留了图像的"上半部 / 下半部 + 4 列"这种粗粒度的空间结构——这正是 QCNN 卷积层"邻居比特相关"假设能利用的先验。


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))

# 上排：4 张原始 8x8 图
for i in range(4):
    axes[0, i].imshow(data.raw_train_images[i], cmap="gray_r", interpolation="nearest")
    axes[0, i].set_title(f"raw 8x8\nlabel={data.y_train[i]}", fontsize=9)
    axes[0, i].set_xticks([]); axes[0, i].set_yticks([])

# 下排：池化后的 2x4 矩阵
pooled = data.x_train.reshape(-1, 2, 4)  # (N, 2, 4)
for i in range(4):
    axes[1, i].imshow(pooled[i], cmap="RdBu", interpolation="nearest", vmin=-np.pi, vmax=np.pi)
    axes[1, i].set_title(f"pooled 2x4 (-π~π)", fontsize=9)
    axes[1, i].set_xticks([]); axes[1, i].set_yticks([])

fig.suptitle("原始 vs 池化后", fontsize=11)
fig.tight_layout()
plt.show()


## 3. 构建 8 比特 QCNN

电路拓扑：

```
z_feature_map(8, reps=2)
  -> conv1 (8 qubits)  -> pool1 (8 -> 4)
  -> conv2 (4 qubits)  -> pool2 (4 -> 2)
  -> conv3 (2 qubits)  -> pool3 (2 -> 1)
```

最后测量末位比特（q7）的 Pauli-Z 期望，输出范围 [-1, 1]，恰好可以让标签 {-1, +1} 用平方误差直接拟合。


In [ ]:
from src.model import build_qcnn

bundle = build_qcnn(num_qubits=8, feature_map_kind="z")
n_weights = len(bundle.ansatz.parameters)
print(f"ansatz 可训练参数数: {n_weights}")

# 画 ansatz（折叠成单元方便看）
fig = bundle.ansatz.draw("mpl", style="iqp", fold=80)
fig.set_size_inches(14, 6)
fig


In [ ]:
viz.plot_circuit(bundle.full_circuit.decompose(), out_path=OUT_DIR / "circuit.png")
print("完整电路（feature_map + ansatz 解构后）已保存:", OUT_DIR / "circuit.png")


## 4. 训练 QCNN

- 优化器：`SPSA(maxiter=150)` —— 每步只需 2 次电路求值（与参数维度无关），高维 + noisy 友好
- 损失：`squared_error`（输出 ∈ [-1, 1] 与 ±1 标签匹配）
- 标签映射：{0, 1} → {-1, +1}
- 初始权重：均匀分布 [-0.5, 0.5]，避开 barren plateau


In [ ]:
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import SPSA

loss_history = []

def callback(*args):
    # 兼容 SciPy 优化器 (weights, loss) 与 SPSA (nfev, params, loss, stepsize, accepted)
    loss = args[1] if len(args) == 2 else args[2]
    loss_history.append(float(loss))
    if len(loss_history) % 10 == 0 or len(loss_history) == 1:
        print(f"  iter {len(loss_history):3d}  loss = {float(loss):.4f}")

initial_point = (algorithm_globals.random.random(n_weights) - 0.5) * 1.0

classifier = NeuralNetworkClassifier(
    neural_network=bundle.qnn,
    loss="squared_error",
    optimizer=SPSA(maxiter=150),
    callback=callback,
    initial_point=initial_point,
)

# {0, 1} -> {-1, +1}
y_train_pm = 2 * data.y_train - 1
y_test_pm = 2 * data.y_test - 1

t0 = time.perf_counter()
classifier.fit(data.x_train, y_train_pm)
qcnn_time = time.perf_counter() - t0
print(f"\n训练完成，用时 {qcnn_time:.1f} 秒，迭代 {len(loss_history)} 次")


In [ ]:
viz.plot_training_curve(loss_history, out_path=OUT_DIR / "training_curve.png")
Image(filename=str(OUT_DIR / "training_curve.png"))


## 5. 评估：训练/测试准确率、混淆矩阵、错分样本

In [ ]:
qcnn_train_acc = float(classifier.score(data.x_train, y_train_pm))
qcnn_test_acc = float(classifier.score(data.x_test, y_test_pm))
print(f"QCNN  train acc = {qcnn_train_acc:.4f}")
print(f"QCNN  test  acc = {qcnn_test_acc:.4f}")

y_pred_pm = classifier.predict(data.x_test)
y_pred = ((np.asarray(y_pred_pm).ravel() + 1) // 2).astype(np.int64)

viz.plot_confusion_matrix(
    data.y_test, y_pred, classes=("0", "1"), out_path=OUT_DIR / "confusion_matrix.png"
)
Image(filename=str(OUT_DIR / "confusion_matrix.png"))


In [ ]:
has_misclf = viz.plot_misclassified(
    data.raw_test_images, data.y_test, y_pred, max_n=6,
    out_path=OUT_DIR / "misclassified.png",
)
if has_misclf:
    display(Image(filename=str(OUT_DIR / "misclassified.png")))
else:
    print("测试集上没有错分样本，QCNN 完美区分了 0 和 1。")


## 6. 模型保存与加载

`SerializableModelMixin` 提供 `to_dill / from_dill`（替代已废弃的 `save / load`）。
保存的是整个 classifier 对象（含 QNN / 权重 / primitive）。


In [ ]:
model_path = OUT_DIR / "qcnn_model.dill"
classifier.to_dill(str(model_path))

loaded = NeuralNetworkClassifier.from_dill(str(model_path))
reloaded_acc = float(loaded.score(data.x_test, y_test_pm))
print(f"原模型 acc:   {qcnn_test_acc:.4f}")
print(f"加载模型 acc: {reloaded_acc:.4f}")
assert abs(qcnn_test_acc - reloaded_acc) < 1e-9, "保存/加载后准确率不一致！"
print("OK：保存与加载得到完全相同的准确率。")


## 7. 经典基线对比

为了让对比公平：经典 SVM / MLP 使用**完全相同的 8 维池化特征**作为输入，只是不再走量子电路。


In [ ]:
from src.classical_baselines import train_svm, train_mlp

svm_res = train_svm(data.x_train, data.y_train, data.x_test, data.y_test, seed=SEED)
mlp_res = train_mlp(data.x_train, data.y_train, data.x_test, data.y_test, seed=SEED)

for r in (svm_res, mlp_res):
    print(f"{r.name:14s}  train_acc={r.train_acc:.4f}  test_acc={r.test_acc:.4f}  time={r.train_time:.3f}s")

results = {
    "QCNN":          {"test_acc": qcnn_test_acc, "train_time": qcnn_time},
    svm_res.name:    {"test_acc": svm_res.test_acc, "train_time": svm_res.train_time},
    mlp_res.name:    {"test_acc": mlp_res.test_acc, "train_time": mlp_res.train_time},
}

viz.plot_comparison(results, out_path=OUT_DIR / "comparison.png")
Image(filename=str(OUT_DIR / "comparison.png"))


## 8. 总结与延伸

### 典型结果

| 模型 | 测试准确率 | 训练时间 |
|------|-----------|---------|
| QCNN (SPSA, 150 iter) | ~85–95% | ~5–8 分钟 |
| SVM (RBF) | ~100% | < 0.1 秒 |
| MLP (16-8) | ~90–100% | ~0.1 秒 |

对 0/1 这种线性可分性极强的任务，经典模型几乎无需训练就能完美区分。
QCNN 在精度上能逼近经典模型，但训练成本（电路求值次数）高出几个数量级。
**这正是当前 NISQ 阶段的现状**：量子优势主要体现在结构化问题或高维特征空间中，简单图像并不是它的主战场。

### 关键观察：数据的"空间结构"对 QCNN 至关重要

我们用 `mode="pca"` 跑过对照实验，loss 几乎不下降、acc 卡在 ~60%。
原因：QCNN 的卷积层假设"邻居比特之间存在相关性"，
而 PCA 主成分相互独立、没有空间局部性，QCNN 无从利用先验。
所以 demo 默认采用 8×8 → 2×4 平均池化，保留行/列粗粒度的空间结构。

### 进一步玩法

- **改成 0/1/2 三分类**：用 OneHot encoding，QNN 输出多个 observable 的期望值，loss 改 `cross_entropy`。
- **换 ansatz**：用 `qiskit.circuit.library.real_amplitudes`、`efficient_su2` 等更通用的电路。
- **换优化器**：`COBYLA`、`L_BFGS_B`、`ADAM`，对比收敛速度（脚本支持 `--optimizer` 参数）。
- **加噪声模拟**：用 `qiskit_aer.AerSimulator` 接入 `EstimatorQNN(estimator=...)`，看噪声鲁棒性。
- **混合模型**：用 `TorchConnector` 把 QCNN 嵌入 PyTorch CNN 的最后一层，前面接经典卷积层提取特征。
- **真实硬件**：用 `qiskit-ibm-runtime` 的 `EstimatorV2` 替换默认 estimator，跑在 IBM 量子计算机上。
